# Exercício Extra 02 — Classificando fotos reais das letras A e I

**Objetivo:** usar o KNN treinado no **Exercício Extra 01** para classificar fotos
tiradas no celular das letras A e I escritas à mão no papel.

**Passo a passo (fora do notebook):**
1. Escreva as letras **A** e **I** em um papel (uma de cada vez, com caneta/lápis bem visível)
2. Tire uma foto de cada letra com o celular
3. Transfira as fotos para o computador e salve dentro da pasta `imgs/exercicio_extra_02/`
   (ex: `imgs/exercicio_extra_02/teste_A.jpg`, `imgs/exercicio_extra_02/teste_I.jpg`)
4. Rode as células abaixo

**Importante:** este notebook depende do arquivo `modelo_knn_treinado.npz`, gerado ao final
do `exercicio_extra_01.ipynb`. Rode aquele notebook primeiro (ou pelo menos a última célula
dele) caso o arquivo ainda não exista.

## 0. Instalação das dependências

In [ ]:
%pip install -q numpy matplotlib pillow

## 1. Carregando o modelo treinado no Extra 01

O "modelo" do KNN é apenas o conjunto de treino + o melhor K encontrado pela validação
cruzada — por isso basta carregar o `.npz` salvo, sem precisar retreinar nada.

In [ ]:
import numpy as np

modelo = np.load("modelo_knn_treinado.npz")

X_treino = modelo["X_treino"]
y_treino = modelo["y_treino"]
melhor_k = int(modelo["melhor_k"])
IMG_HEIGHT = int(modelo["img_height"])
IMG_WIDTH = int(modelo["img_width"])

print(f"Modelo carregado: {X_treino.shape[0]} amostras de treino | K = {melhor_k}")
print(f"Dimensão esperada da imagem: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Classes do treino: {np.unique(y_treino)}")

## 2. Implementação do KNN

Mesma implementação (sem bibliotecas de ML) usada no Extra 01 — repetida aqui para este
notebook funcionar de forma independente.

In [ ]:
def distancia_euclidiana(X_treino, x_consulta):
    return np.sqrt(np.sum((X_treino - x_consulta) ** 2, axis=1))

def knn_predict_um(x_consulta, X_treino, y_treino, k):
    distancias = distancia_euclidiana(X_treino, x_consulta)
    idx_vizinhos = np.argsort(distancias)[:k]
    rotulos_vizinhos = y_treino[idx_vizinhos]
    valores, contagens = np.unique(rotulos_vizinhos, return_counts=True)
    return valores[np.argmax(contagens)]

def knn_predict(X_consulta, X_treino, y_treino, k):
    return np.array([knn_predict_um(x, X_treino, y_treino, k) for x in X_consulta])

## 3. Pré-processamento das fotos

As fotos do celular têm formato/cores diferentes das imagens de treino (EMNIST). Para o
KNN funcionar, cada foto precisa virar um vetor **na mesma escala e dimensão** dos dados
de treino:

1. Converter para escala de cinza
2. Redimensionar para `IMG_HEIGHT x IMG_WIDTH` (mesma dimensão do treino)
3. **Inverter as cores se necessário** — o EMNIST representa o traço da letra em
   branco sobre fundo preto; uma foto normal é o oposto (traço escuro em fundo claro).
   Se as previsões saírem ruins, esse é o primeiro ponto a conferir/ajustar.
4. Ajustar brilho/contraste (opcional, ajuda se a foto estiver muito clara/escura)
5. Normalizar os pixels para `[0, 1]`, igual ao treino

In [ ]:
from PIL import Image, ImageOps

def preprocessar_foto(caminho_imagem, inverter_cores=True, ajuste_brilho=0):
    img = Image.open(caminho_imagem).convert("L")              # escala de cinza
    img = img.resize((IMG_WIDTH, IMG_HEIGHT))                    # mesma dimensão do treino

    if inverter_cores:
        img = ImageOps.invert(img)                               # traço claro / fundo escuro, como no EMNIST

    arr = np.array(img, dtype=np.float64)

    if ajuste_brilho != 0:
        arr = np.clip(arr + ajuste_brilho, 0, 255)

    arr = arr / 255.0                                            # normaliza, igual ao treino
    return arr.flatten()

## 4. Classificando as fotos

Carrega todas as imagens da pasta `imgs/exercicio_extra_02/`, pré-processa cada uma e
classifica com o KNN treinado. Mostra a foto original lado a lado com a predição.

In [ ]:
import glob
import os
import matplotlib.pyplot as plt

PASTA_FOTOS = "imgs/exercicio_extra_02"
caminhos_fotos = sorted(
    glob.glob(os.path.join(PASTA_FOTOS, "*.jpg")) +
    glob.glob(os.path.join(PASTA_FOTOS, "*.jpeg")) +
    glob.glob(os.path.join(PASTA_FOTOS, "*.png"))
)

if not caminhos_fotos:
    print(f"Nenhuma foto encontrada em '{PASTA_FOTOS}'. Salve as fotos das letras lá e rode novamente.")
else:
    fig, axes = plt.subplots(1, len(caminhos_fotos), figsize=(4 * len(caminhos_fotos), 4))
    if len(caminhos_fotos) == 1:
        axes = [axes]

    for ax, caminho in zip(axes, caminhos_fotos):
        x_foto = preprocessar_foto(caminho)
        pred = knn_predict_um(x_foto, X_treino, y_treino, melhor_k)

        img_original = Image.open(caminho)
        ax.imshow(img_original, cmap="gray")
        ax.set_title(f"{os.path.basename(caminho)}\nPredição: {pred}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 5. Discussão (pontos para a apresentação)

- O modelo acertou as letras escritas à mão? Se errou, foi por causa da inversão de
  cores, do recorte/enquadramento da foto, ou do traço da caligrafia ser muito diferente
  do estilo do EMNIST?
- Por que é necessário aplicar exatamente o **mesmo pré-processamento** (tamanho, escala
  de cinza, normalização) nas fotos novas que foi aplicado nos dados de treino?
- Esse é um exemplo prático do problema de **generalização**: um modelo treinado em um
  domínio (dataset EMNIST, escrito por outras pessoas, digitalizado de forma padronizada)
  precisa lidar com dados de um domínio ligeiramente diferente (foto de celular, sua
  própria caligrafia). Isso é chamado de *domain shift* — o quanto isso impactou o resultado?